In [1]:
from datetime import datetime
import pandas as pd
import yfinance as yf

# ==============================================================================
# 1. TICKER CONFIGURATION
# ==============================================================================

# Cross-asset portfolio tracking: Crypto, Commodities, Fixed Income, and Global Equities.
# Maintained as a flat list to pass directly to the yfinance batch downloader.
tickers = [
    "BTC-USD", "ETH-USD", "SOL-USD", "LINK-USD", "GC=F", "SI=F", "BZ=F", "NG=F", "HG=F", 
    "ZC=F", "KC=F", "PA=F", "TLT", "IEF", "SHY", "TIP", "BNDX", "EMB", "VTC", "JNK", 
    "IBGL.L", "MUB", "AAPL", "MSFT", "AMZN", "JNJ", "JPM", "XOM", "PG", "TSLA", 
    "UNH", "BRK-B", "SAN.MC", "ITX.MC", "IBE.MC", "MC.PA", "SAP.DE", "ASML.AS", 
    "SIE.DE", "NESN.SW", "AZN.L", "HSBA.L", "2330.TW", "7203.T", "BABA", 
    "TCEHY", "RELIANCE.NS", "VALE", "BHP"
]

# ==============================================================================
# 2. TEMPORAL ANCHORING
# ==============================================================================

# Fetch the current date in ISO format. Because the yfinance 'end' parameter is 
# exclusive, this guarantees that data processing stops right before today's 
# active/incomplete trading session, preventing intraday data contamination.
fecha_hoy = datetime.now().strftime("%Y-%m-%d")

# ==============================================================================
# 3. DATA INGESTION
# ==============================================================================

# Download full historical OHLCV data. 
# `start="1900-01-01"` acts as an effective floor to capture the maximum available history per asset.
# `group_by="ticker"` outputs a MultiIndex column structure (Ticker -> Market Metric).
# `auto_adjust=False` preserves raw, unadjusted close prices to maintain data lineage.
data = yf.download(
    tickers,
    start="1900-01-01",  
    end=fecha_hoy,       
    group_by="ticker",
    auto_adjust=False
)

# ==============================================================================
# 4. OPTIMIZED TRANSFORMATION (WIDE TO LONG FORMAT)
# ==============================================================================

# Pivot the MultiIndex columns from wide format to a clean, long format database structure.
# `level=0` targets the ticker symbol, and `future_stack=True` ensures compliance 
# with upcoming pandas internal behavior modifications.
final_df = data.stack(level=0, future_stack=True)

# Explicitly assign names to the resulting MultiIndex before resetting to guarantee
# seamless column mapping when flattening the index.
final_df.index.names = ["date", "ticker"]
final_df = final_df.reset_index()

# ==============================================================================
# 5. CLEANING & FILTERING
# ==============================================================================

# Enforce strict data completeness. Drop any rows where core price metrics are missing,
# which effectively filters out non-trading days, exchange holidays, and tracking gaps.
final_df = final_df.dropna(subset=["Open", "High", "Low", "Close"])

# ==============================================================================
# 6. UNIQUE ID GENERATION & SORTING
# ==============================================================================

# Construct a composite business key (asset_id) combining the ticker and the ISO date.
# This serves as a reliable primary key for downstream relational databases or analytical pipelines.
final_df["asset_id"] = final_df["ticker"] + "_" + final_df["date"].dt.strftime("%Y-%m-%d")

# Enforce a strict schema layout by explicitly ordering columns, then sort the dataset
# chronologically per asset to ensure structural consistency for time-series operations.
final_df = final_df[
    ["asset_id", "ticker", "date", "Open", "High", "Low", "Close", "Volume"]
].sort_values(["ticker", "date"])

# ==============================================================================
# 7. PERSISTENCE
# ==============================================================================

# Serialize the finalized dataset to a CSV file. `index=False` prevents pandas 
# from writing its redundant, auto-generated sequential row numbers.
final_df.to_csv("data/historical_assets.csv", index=False)

[*********************100%***********************]  49 of 49 completed
